In [2]:
from TDNN import *
import TDNN as TDNN
from TDNNF import *
import TDNNF as TDNNF
import sys
import pandas as pd
import os, re

In [ ]:
CSV = r"D:\118_Rio\118\src\codeStat\preModelling\out\dataBersih.csv"
OUT = r"D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\makeAligner\sampleddata\HasilLexicon.csv"

_PH = {
    "ng":"ng","ny":"ny","th":"th","dh":"dh",
    "a":"a","b":"b","c":"ch","d":"d","e":"e","f":"f","g":"g","h":"h",
    "i":"i","j":"j","k":"k","l":"l","m":"m","n":"n","o":"o","p":"p",
    "r":"r","s":"s","t":"t","u":"u","v":"v","w":"w","y":"y","z":"z",
    "x":"k","q":"k",
}

_MJ = {"Ã©":"é","Ã¨":"è","Ã ":"à","Ã¢":"â","Ã®":"î","Ã´":"ô",
       "Ã»":"û","Ã«":"ë","Ã¯":"ï","Ã¼":"ü","Ã±":"ñ","Ã§":"ç"}
_AC = str.maketrans("éèêëàáâäîïíìôóòöûúùüñç", "eeeeaaaaiiiioooouuuunc")


def normalise(text):
    for b, g in _MJ.items():
        text = text.replace(b, g)
    text = text.lower().strip()
    text = re.sub(r'["\'`\u201c\u201d\u2018\u2019()\[\]{}]', " ", text)
    text = re.sub(r"[+\-*/=<>%^&|~\\@#$.,;:!?]", " ", text)
    text = text.translate(_AC)
    text = re.sub(r"[^a-z\s]", "", text)
    return re.sub(r"\s+", " ", text).strip()


def word_to_phones(word):
    out, i = [], 0
    while i < len(word):
        if i+1 < len(word) and word[i:i+2] in _PH:
            out.append(_PH[word[i:i+2]]); i += 2
        elif word[i] in _PH:
            out.append(_PH[word[i]]); i += 1
        else:
            i += 1
    return out


df = pd.read_csv(CSV)
vocab = set()
for sent in df["sentences"].dropna():
    for word in normalise(str(sent)).split():
        vocab.add(word)

print(f"Total vocab: {len(vocab)} kata")

rows = []
for word in sorted(vocab):
    phones = word_to_phones(word)
    if phones:
        rows.append({"Kata": word, "Fonem": " ".join(phones)})

os.makedirs(os.path.dirname(OUT), exist_ok=True)
pd.DataFrame(rows).to_csv(OUT, index=False, encoding="utf-8")
print(f"Lexicon selesai: {len(rows)} kata")
print(pd.read_csv(OUT).head(10).to_string())

Total vocab: 20144 kata
Lexicon selesai: 20144 kata
         Kata                Fonem
0           a                    a
1          aa                  a a
2         aak                a a k
3     aalborg        a a l b o r g
4    aalsmeer      a a l s m e e r
5         aan                a a n
6  aaosronson  a a o s r o n s o n
7       aaron            a a r o n
8          ab                  a b
9       ababa            a b a b a


In [17]:

# Load data
df = pd.read_csv(r"D:\118_Rio\118\src\codeStat\preModelling\out\dataBersih.csv")

# Buat kolom audio_id dari Path_Data
df["audio_id"] = df["Path_Data"].apply(lambda p: os.path.splitext(os.path.basename(str(p)))[0])

# Shuffle
df = df.sample(frac=1, random_state=42).reset_index(drop=True)

# Split 80% train, 20% test
split_idx = int(len(df) * 0.8)
df["split"] = "test"
df.loc[:split_idx-1, "split"] = "train"

print(f"Total: {len(df)} | Train: {len(df[df.split=='train'])} | Test: {len(df[df.split=='test'])}")
print(df[["audio_id", "split"]].head())

# Simpan
out_dir = r"D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN\out"
os.makedirs(out_dir, exist_ok=True)
df.to_csv(os.path.join(out_dir, "HasilTrainTestSplit.csv"), index=False)

print("HasilTrainTestSplit.csv berhasil dibuat")

Total: 17880 | Train: 14304 | Test: 3576
     audio_id  split
0  43cb4f10c7  train
1  9ef9d12f53  train
2  b0c772a941  train
3  cf1e6d2184  train
4  ee8d07f9dd  train
HasilTrainTestSplit.csv berhasil dibuat


In [ ]:
# import pandas as pd
# import os

# CSV = r"D:\118_Rio\118\src\codeStat\preModelling\out\dataBersih.csv"

# df = pd.read_csv(CSV)

# # Ambil hanya filename, gabung ke folder flat
# df["Path_Data"] = df["Path_Data"].apply(
#     lambda p: os.path.join(r"D:\118_Rio\118\datasampled", os.path.basename(str(p)))
# )

# print("Contoh path baru:")
# print(df["Path_Data"].head(3).to_string())

# exists = df["Path_Data"].apply(lambda p: os.path.exists(str(p))).sum()
# print(f"\nFile ditemukan: {exists}/{len(df)}")

# # Simpan
# df.to_csv(CSV, index=False)
# print(f"Disimpan ke: {CSV}")

Contoh path baru:
0    D:\118_Rio\118\datasampled\00e1f837f4.wav
1    D:\118_Rio\118\datasampled\002762f6a1.wav
2    D:\118_Rio\118\datasampled\00079b49f9.wav

File ditemukan: 17880/17880
Disimpan ke: D:\118_Rio\118\src\codeStat\preModelling\out\dataBersih.csv


In [2]:
from Shared_resource import prepare_mfa_label_map
prepare_mfa_label_map(
    tg_dir=r'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid',
    label_map_out='label_map_fixed.csv'
)

[23:52:35] Label map dari TextGrid: 29 fonem unik
[23:52:35] Urutan fonem: ['a', 'e', 'i', 'o', 'u', 'b', 'ch', 'd', 'dh', 'f', 'g', 'h', 'j', 'k', 'ks', 'l', 'm', 'n', 'ng', 'ny', 'p', 'r', 's', 't', 'th', 'v', 'w', 'y', 'z']
[23:52:35] Label map disimpan: label_map_fixed.csv  (29 fonem)


{'a': 0,
 'e': 1,
 'i': 2,
 'o': 3,
 'u': 4,
 'b': 5,
 'ch': 6,
 'd': 7,
 'dh': 8,
 'f': 9,
 'g': 10,
 'h': 11,
 'j': 12,
 'k': 13,
 'ks': 14,
 'l': 15,
 'm': 16,
 'n': 17,
 'ng': 18,
 'ny': 19,
 'p': 20,
 'r': 21,
 's': 22,
 't': 23,
 'th': 24,
 'v': 25,
 'w': 26,
 'y': 27,
 'z': 28}

In [2]:
sys.argv = [sys.argv[0], "--mode", "train", "--epochs", "40"]
TDNN.main()

[17:23:25] TDNN ASR Bahasa Jawa — mode=train  device=cuda
[17:23:25] Manifest sudah ada, skip build_manifests
[17:23:25] AlignmentBridge (MFA): 29 kelas fonem
[17:23:26] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[17:23:26] LDA: 27 dims
[17:23:26] INPUT_DIM = 27
[17:23:27] Lexicon: 20,120 entri kata (24 loopback dilewati)
[17:23:28] p2w: 20,108 entri (24 loopback dilewati)
[17:23:29] NgramLM(3, KN d=0.75): vocab=20,147, N=177,777
[17:23:29] NgramLM disimpan: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN\out\lm.pkl
[17:23:29] build_transition_matrix: memproses 17,858 TextGrid ...
[17:23:37] build_transition_matrix: 675,357 bigram fonem ditemukan
[17:23:37] build_transition_matrix: selesai  shape=(29, 29)  min=-18.42  max=-0.74
[17:23:37] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[17:23:37] Dataset manifest_val.jsonl: 3,571 utts (dari 3

In [3]:
sys.argv = [sys.argv[0]]  
sys.argv += ["--mode", "train", "--epochs", "40"]
TDNNF.main()

[12:13:38] TDNN-F ASR Bahasa Jawa — mode=train  device=cuda
[12:13:38] Manifest sudah ada, skip build_manifests
[12:13:38] AlignmentBridge (MFA): 30 kelas fonem
[12:13:39] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[12:13:39] LDA: 27 dims
[12:13:39] INPUT_DIM = 27
[12:13:40] Lexicon: 20,120 entri kata (24 loopback dilewati)
[12:13:41] p2w: 20,108 entri (24 loopback dilewati)
[12:13:41] build_transition_matrix: memproses 17,858 TextGrid ...
[12:13:49] build_transition_matrix: 675,357 bigram fonem ditemukan
[12:13:49] build_transition_matrix: selesai  shape=(30, 30)  min=-18.42  max=0.00
[12:13:49] NgramLM dimuat: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN-F\out\lm.pkl
[12:13:49] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[12:13:49] Dataset manifest_val.jsonl: 3,571 utts (dari 3,576 total, 5 tanpa TextGrid)
[12:13:49] TDNN-F: 12 blok 

In [2]:
sys.argv = [sys.argv[0]]  
sys.argv += ["--mode", "train", "--epochs", "40"]
TDNNF.main()

[18:59:48] TDNN-F ASR Bahasa Jawa — mode=train  device=cuda
[18:59:48] Manifest sudah ada, skip build_manifests
[18:59:48] AlignmentBridge (MFA): 30 kelas fonem
[18:59:48] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[18:59:48] LDA: 27 dims
[18:59:48] INPUT_DIM = 27
[18:59:50] NgramLM(3): vocab=20,147, N=177,777
[18:59:50] NgramLM disimpan: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN-F\out\lm.pkl
[18:59:52] build_fuzzy_trie: 20117 kata, 103129 variasi, total 123246 entri trie
[18:59:52] build_transition_matrix: memproses 17,858 TextGrid ...
[18:59:58] build_transition_matrix: 675,357 bigram fonem ditemukan
[18:59:58] build_transition_matrix: selesai shape=(30, 30) min=-18.42 max=0.00
[18:59:58] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[18:59:58] Dataset manifest_val.jsonl: 3,571 utts (dari 3,576 total, 5 tanpa TextGrid)
[18:59:58] TD

In [3]:
sys.argv = [sys.argv[0]]  
sys.argv += ["--mode", "train", "--epochs", "40"]
TDNN.main()

[09:35:49] TDNN ASR Bahasa Jawa — mode=train  device=cuda
[09:35:51] manifest_train.jsonl: 14,304 utts
[09:35:51] manifest_val.jsonl: 3,576 utts
[09:35:51] AlignmentBridge (MFA): 30 kelas fonem
[09:35:51] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[09:35:51] LDA: 27 dims
[09:35:51] INPUT_DIM = 27
[09:35:52] NgramLM(3): vocab=20,147, N=177,777
[09:35:52] NgramLM disimpan: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN\out\lm.pkl
[09:35:54] build_fuzzy_trie: 20117 kata, 103129 variasi, total 123246 entri trie
[09:35:54] build_transition_matrix: memproses 17,858 TextGrid ...
[09:36:01] build_transition_matrix: 675,357 bigram fonem ditemukan
[09:36:01] build_transition_matrix: selesai shape=(30, 30) min=-18.42 max=0.00
[09:36:01] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[09:36:01] Dataset manifest_val.jsonl: 3,571 utts (dari 3,576 total, 

In [5]:
sys.argv = [sys.argv[0]]  
sys.argv += ["--mode", "train", "--epochs", "100"]
TDNNF.main()

[13:19:19] TDNN-F ASR Bahasa Jawa — mode=train  device=cuda
[13:19:19] Manifest sudah ada, skip build_manifests
[13:19:19] AlignmentBridge (MFA): 30 kelas fonem
[13:19:19] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[13:19:19] LDA: 27 dims
[13:19:19] INPUT_DIM = 27
[13:19:20] NgramLM dimuat: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN-F\out\lm.pkl
[13:20:02] build_fuzzy_trie: 20117 kata, 103129 variasi, total 123246 entri trie
[13:20:02] build_transition_matrix: memproses 17,858 TextGrid ...
[13:20:08] build_transition_matrix: 675,357 bigram fonem ditemukan
[13:20:08] build_transition_matrix: selesai shape=(30, 30) min=-18.42 max=0.00
[13:20:09] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[13:20:09] Dataset manifest_val.jsonl: 3,571 utts (dari 3,576 total, 5 tanpa TextGrid)
[13:20:09] TDNN-F: 12 blok  h=512  btn=128  bypass=0.66  in=27

In [ ]:
sys.argv = [sys.argv[0], "--mode", "tune", 
            "--search","random",
            "--ept","70"]
TDNN.main()

[09:35:24] TDNN ASR Bahasa Jawa — mode=tune  device=cuda
[09:35:24] Manifest sudah ada, skip build_manifests
[09:35:24] AlignmentBridge (MFA): 30 kelas fonem
[09:35:24] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[09:35:24] LDA: 27 dims
[09:35:24] INPUT_DIM = 27
[09:35:24] NgramLM dimuat: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN\out\lm.pkl
[09:35:27] build_fuzzy_trie: 20117 kata, 103129 variasi, total 123246 entri trie
[09:35:27] build_transition_matrix: memproses 17,858 TextGrid ...
[09:35:34] build_transition_matrix: 675,357 bigram fonem ditemukan
[09:35:34] build_transition_matrix: selesai shape=(30, 30) min=-18.42 max=0.00
[09:35:34] Tuning TDNN — 10 trial x 70 epoch (random)
[09:35:34] Trial 1/10: L=6 H=256 lr=0.0005 drop=0.1
[09:35:35] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[09:35:35] Dataset manifest_val.jsonl: 3,571 utt

In [ ]:
sys.argv = [sys.argv[0]]  
sys.argv += [ "--mode", "tune", 
            "--ept", "70",
            "--trials","10",
            "--search","random"]
TDNNF.main()

[17:32:11] TDNN-F ASR Bahasa Jawa — mode=tune  device=cuda
[17:32:11] Manifest sudah ada, skip build_manifests
[17:32:11] AlignmentBridge (MFA): 30 kelas fonem
[17:32:11] TextGrid ditemukan: 17,858 file di 'D:\118_Rio\118\src\codeStat\modelling\voiceModelling\kaldialignment\MFA\sampleddata\mfa_textgrid'
[17:32:11] LDA: 27 dims
[17:32:11] INPUT_DIM = 27
[17:32:11] NgramLM dimuat: D:\118_Rio\118\src\codeStat\modelling\voiceModelling\TDNN-TDNN-F\TDNN-F\out\lm.pkl
[17:32:13] build_fuzzy_trie: 20117 kata, 103129 variasi, total 123246 entri trie
[17:32:13] build_transition_matrix: memproses 17,858 TextGrid ...
[17:32:19] build_transition_matrix: 675,357 bigram fonem ditemukan
[17:32:19] build_transition_matrix: selesai shape=(30, 30) min=-18.42 max=0.00
[17:32:19] Tuning TDNN-F — 10 trial x 70 epoch (random)
[17:32:19] Trial 1/10: blk=12  btn=128  bypass=0.33  lr=0.0005  drop=0.1
[17:32:19] Dataset manifest_train.jsonl: 14,287 utts (dari 14,304 total, 17 tanpa TextGrid)
[17:32:19] Dataset ma